<a href="https://colab.research.google.com/github/rathorebharat/mftracker/blob/main/mf-correction-engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# =================================================================================================
# MUTUAL FUND CORRECTION + PROFIT + INVESTMENT ENGINE
#
# VERSION: V8.4
#
# ARCHITECTURE
# -------------------------------------------------------------------------------------------------
# SECTION 1 : Fetch / prepare NAV data for ALL funds
# SECTION 2 : Correction Episode Engine
# SECTION 3 : Diagnostics 1 - Current correction + historical opportunities
# SECTION 4 : Diagnostics 2 - Profit taking + overshoot + strategic group overlap
# SECTION 5 : Investment Engine - allocation of ₹2,00,000
#
# IMPORTANT DESIGN PRINCIPLES
# -------------------------------------------------------------------------------------------------
# 1. NAV history already contains the latest NAV.
#    There is NO separate "latest NAV" dataset.
#
# 2. Corrections are measured as PEAK -> TROUGH episodes.
#    Daily observations are NOT treated as separate correction opportunities.
#
# 3. Historical correction percentiles are calculated from COMPLETED episodes.
#
# 4. The current OPEN correction is never treated as a completed historical episode.
#
# 5. Profit-taking is an independent engine.
#
# 6. Historical-trough growth is an independent engine.
#
# 7. Strategic groups are used to detect overlap/concentration.
#
# 8. HDFC Nifty 50 and ICICI Prudential Nifty 50 are considered the SAME
#    strategic exposure for group aggregation.
#
# 9. The four banking funds below retain their individual fund names but
#    belong to the same BANKING_FINANCIAL strategic group.
#
# 10. The final investment amount is a DECISION LAYER, not a replacement
#     for the diagnostic engines.
# =================================================================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)


# -------------------------------------------------------------------------------------------------
# GLOBAL CONFIGURATION
# -------------------------------------------------------------------------------------------------

MIN_CORRECTION_PCT = 3.0

# Recovery is considered complete when NAV gets sufficiently close to
# the previous peak.
RECOVERY_THRESHOLD = 0.995

# When comparing today's correction with an historical episode,
# this tolerance defines "approximately the same depth".
NEAR_TODAY_TOLERANCE = 1.0

# Historical trough must be at least this old before being used
# by the historical-trough-growth engine.
MIN_TROUGH_AGE_DAYS = 180

# Investment amount
TOTAL_MONTHLY_INVESTMENT = 200000


# -------------------------------------------------------------------------------------------------
# ASSET-CLASS SPECIFIC CORRECTION PARAMETERS
#
# These are intentionally conservative.
#
# You can tune them later after validating the diagnostics.
# -------------------------------------------------------------------------------------------------

ASSET_THRESHOLDS = {

    "CORE_EQUITY": {
        "watch": 5.0,
        "accumulate": 8.0,
        "strong_accumulate": 12.0,
    },

    "GROWTH_EQUITY": {
        "watch": 5.0,
        "accumulate": 8.0,
        "strong_accumulate": 15.0,
    },

    "FACTOR_EQUITY": {
        "watch": 5.0,
        "accumulate": 8.0,
        "strong_accumulate": 14.0,
    },

    "SECTOR_EQUITY": {
        "watch": 5.0,
        "accumulate": 8.0,
        "strong_accumulate": 15.0,
    },

    "INTERNATIONAL": {
        "watch": 5.0,
        "accumulate": 8.0,
        "strong_accumulate": 15.0,
    },

    "HYBRID_DEBT_OTHER": {
        "watch": 3.0,
        "accumulate": 5.0,
        "strong_accumulate": 8.0,
    },

    "DEBT": {
        "watch": 2.0,
        "accumulate": 3.0,
        "strong_accumulate": 5.0,
    },

    "DEFAULT": {
        "watch": 5.0,
        "accumulate": 8.0,
        "strong_accumulate": 15.0,
    }
}

In [24]:
# =================================================================================================
# SECTION 1
# FETCH / PREPARE OVERALL NAV DATA
# =================================================================================================

# -----------------------------------------------------------------------------------------------
# IMPORTANT
# -----------------------------------------------------------------------------------------------
# If you already have df_nav containing:
#
#   Scheme_Code
#   Date
#   NAV
#
# then this section starts directly from that dataframe.
#
# The dataframe should contain the complete historical dataset INCLUDING
# the latest available NAV.
# -----------------------------------------------------------------------------------------------


def prepare_nav_data(df_nav):

    df = df_nav.copy()

    # Normalize column names
    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    required_columns = {
        "Scheme_Code",
        "Date",
        "NAV"
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    # Date
    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    # NAV
    df["NAV"] = pd.to_numeric(
        df["NAV"],
        errors="coerce"
    )

    # Scheme code
    df["Scheme_Code"] = pd.to_numeric(
        df["Scheme_Code"],
        errors="coerce"
    )

    # Remove invalid rows
    df = df.dropna(
        subset=[
            "Scheme_Code",
            "Date",
            "NAV"
        ]
    )

    # NAV must be positive
    df = df[
        df["NAV"] > 0
    ].copy()

    df["Scheme_Code"] = (
        df["Scheme_Code"]
        .astype(int)
    )

    # Remove duplicate NAV observations
    df = (
        df
        .sort_values(
            [
                "Scheme_Code",
                "Date"
            ]
        )
        .drop_duplicates(
            subset=[
                "Scheme_Code",
                "Date"
            ],
            keep="last"
        )
        .reset_index(drop=True)
    )

    return df


# -------------------------------------------------------------------------------------------------
# FUND NAME MAP
#
# Explicitly preserve the names requested by you.
# -------------------------------------------------------------------------------------------------

FUND_NAME_OVERRIDES = {

    109445:
        "ICICI Prudential Banking & Financial Services",

    120244:
        "ICICI Prudential Banking & Financial Services Direct",

    120733:
        "UTI Banking & Financial Services",

    148623:
        "Mirae Asset Banking & Financial Services",
}


def add_fund_names(df):

    df = df.copy()

    if "Scheme_Name" not in df.columns:

        df["Scheme_Name"] = "NoName"

    df["Scheme_Name"] = (
        df["Scheme_Name"]
        .fillna("NoName")
        .astype(str)
    )

    # Apply explicit names
    for code, name in FUND_NAME_OVERRIDES.items():

        mask = (
            df["Scheme_Code"] == code
        )

        df.loc[
            mask,
            "Scheme_Name"
        ] = name

    return df


# -------------------------------------------------------------------------------------------------
# RUN SECTION 1
# -------------------------------------------------------------------------------------------------

df_nav = prepare_nav_data(
    df_nav
)

df_nav = add_fund_names(
    df_nav
)


# -------------------------------------------------------------------------------------------------
# QUICK DATA VALIDATION
# -------------------------------------------------------------------------------------------------

print("=" * 110)
print("SECTION 1 — NAV DATA VALIDATION")
print("=" * 110)

print(
    f"Rows              : {len(df_nav):,}"
)

print(
    f"Funds             : {df_nav['Scheme_Code'].nunique():,}"
)

print(
    f"First NAV date    : {df_nav['Date'].min().date()}"
)

print(
    f"Latest NAV date   : {df_nav['Date'].max().date()}"
)

print()

latest_check = (
    df_nav
    .sort_values("Date")
    .groupby("Scheme_Code")
    .tail(1)
    [
        [
            "Scheme_Code",
            "Scheme_Name",
            "Date",
            "NAV"
        ]
    ]
    .sort_values("Scheme_Code")
)

print(
    latest_check.to_string(
        index=False
    )
)

SECTION 1 — NAV DATA VALIDATION
Rows              : 72,369
Funds             : 26
First NAV date    : 2013-01-01
Latest NAV date   : 2026-08-27

 Scheme_Code                                                               Scheme_Name       Date      NAV
      118825                         Mirae Asset Large Cap Fund - Direct Plan - Growth 2026-08-27 128.5680
      118834                    Mirae Asset Large & Midcap Fund - Direct Plan - Growth 2026-08-27 181.1370
      118989                           HDFC Mid Cap Fund - Growth Option - Direct Plan 2026-08-27 236.9190
      119063                                    HDFC Nifty 50 Index Fund - Direct Plan 2026-08-27 235.7810
      119151                      ESSEL FLEXIBLE INCOME FUND-DIRECT PLAN-GROWTH OPTION 2018-08-20  14.9468
      119218                           DSP Large & Mid Cap Fund - Direct Plan - Growth 2026-08-27 705.6970
      119544          Aditya Birla Sun Life ELSS Tax Saver Fund - Growth - Direct Plan 2026-08-27  70.0500

In [25]:
# =================================================================================================
# SECTION 2
# CORRECTION EPISODE ENGINE
# =================================================================================================


def build_correction_episodes(
    fund_df,
    min_correction_pct=MIN_CORRECTION_PCT,
    recovery_threshold=RECOVERY_THRESHOLD
):
    """
    Build independent Peak -> Trough -> Recovery episodes.

    A correction episode is counted ONCE.

    Daily NAV movements inside the same correction are NOT separate episodes.
    """

    df = (
        fund_df
        .sort_values("Date")
        .reset_index(drop=True)
        .copy()
    )

    if len(df) < 3:
        return pd.DataFrame()

    nav = df["NAV"].astype(float).values
    dates = df["Date"].values

    episodes = []

    peak_idx = 0
    trough_idx = 0

    in_correction = False

    for i in range(1, len(df)):

        current_nav = nav[i]

        # -----------------------------------------------------------------------------------------
        # NO ACTIVE CORRECTION
        # -----------------------------------------------------------------------------------------

        if not in_correction:

            # New all-time high
            if current_nav >= nav[peak_idx]:

                peak_idx = i
                trough_idx = i

                continue

            correction = (
                (nav[peak_idx] - current_nav)
                / nav[peak_idx]
                * 100
            )

            # Meaningful correction begins
            if correction >= min_correction_pct:

                in_correction = True
                trough_idx = i

            continue

        # -----------------------------------------------------------------------------------------
        # ACTIVE CORRECTION
        # -----------------------------------------------------------------------------------------

        # New lower trough
        if current_nav < nav[trough_idx]:

            trough_idx = i

            continue

        peak_nav = nav[peak_idx]

        recovery_ratio = (
            current_nav / peak_nav
        )

        # -----------------------------------------------------------------------------------------
        # RECOVERY COMPLETE
        # -----------------------------------------------------------------------------------------

        if recovery_ratio >= recovery_threshold:

            trough_nav = nav[trough_idx]

            correction_pct = (
                (peak_nav - trough_nav)
                / peak_nav
                * 100
            )

            if correction_pct >= min_correction_pct:

                peak_date = pd.Timestamp(
                    dates[peak_idx]
                )

                trough_date = pd.Timestamp(
                    dates[trough_idx]
                )

                recovery_date = pd.Timestamp(
                    dates[i]
                )

                recovered_pct = (
                    (current_nav - trough_nav)
                    / (peak_nav - trough_nav)
                    * 100
                )

                episodes.append({

                    "Peak_Date":
                        peak_date,

                    "Peak_NAV":
                        peak_nav,

                    "Trough_Date":
                        trough_date,

                    "Trough_NAV":
                        trough_nav,

                    "Recovery_Date":
                        recovery_date,

                    "Recovery_NAV":
                        current_nav,

                    "Correction_%":
                        correction_pct,

                    "Peak_to_Trough_Days":
                        (
                            trough_date
                            - peak_date
                        ).days,

                    "Trough_to_Recovery_Days":
                        (
                            recovery_date
                            - trough_date
                        ).days,

                    "Total_Recovery_Days":
                        (
                            recovery_date
                            - peak_date
                        ).days,

                    "Recovered_%":
                        recovered_pct,

                    "Open":
                        False
                })

            # Recovery completed.
            # The recovery point becomes the new peak reference.
            peak_idx = i
            trough_idx = i

            in_correction = False

    # ---------------------------------------------------------------------------------------------
    # CURRENT OPEN CORRECTION
    # ---------------------------------------------------------------------------------------------

    if in_correction:

        peak_nav = nav[peak_idx]
        trough_nav = nav[trough_idx]
        current_nav = nav[-1]

        correction_pct = (
            (peak_nav - trough_nav)
            / peak_nav
            * 100
        )

        if correction_pct >= min_correction_pct:

            recovered_pct = (
                (current_nav - trough_nav)
                / (peak_nav - trough_nav)
                * 100
            )

            episodes.append({

                "Peak_Date":
                    pd.Timestamp(
                        dates[peak_idx]
                    ),

                "Peak_NAV":
                    peak_nav,

                "Trough_Date":
                    pd.Timestamp(
                        dates[trough_idx]
                    ),

                "Trough_NAV":
                    trough_nav,

                "Recovery_Date":
                    pd.NaT,

                "Recovery_NAV":
                    np.nan,

                "Correction_%":
                    correction_pct,

                "Peak_to_Trough_Days":
                    (
                        pd.Timestamp(
                            dates[trough_idx]
                        )
                        -
                        pd.Timestamp(
                            dates[peak_idx]
                        )
                    ).days,

                "Trough_to_Recovery_Days":
                    np.nan,

                "Total_Recovery_Days":
                    np.nan,

                "Recovered_%":
                    recovered_pct,

                "Open":
                    True
            })

    episodes_df = pd.DataFrame(
        episodes
    )

    return episodes_df


# -------------------------------------------------------------------------------------------------
# BUILD EPISODES FOR ALL FUNDS
# -------------------------------------------------------------------------------------------------

def build_all_correction_episodes(df_nav):

    all_episodes = []

    for scheme_code, fund_df in df_nav.groupby(
        "Scheme_Code"
    ):

        episodes = build_correction_episodes(
            fund_df
        )

        if episodes.empty:
            continue

        scheme_name = (
            fund_df["Scheme_Name"]
            .iloc[0]
        )

        episodes["Scheme_Code"] = (
            scheme_code
        )

        episodes["Scheme_Name"] = (
            scheme_name
        )

        all_episodes.append(
            episodes
        )

    if not all_episodes:
        return pd.DataFrame()

    result = pd.concat(
        all_episodes,
        ignore_index=True
    )

    return result[
        [
            "Scheme_Code",
            "Scheme_Name",
            "Peak_Date",
            "Peak_NAV",
            "Trough_Date",
            "Trough_NAV",
            "Recovery_Date",
            "Recovery_NAV",
            "Correction_%",
            "Peak_to_Trough_Days",
            "Trough_to_Recovery_Days",
            "Total_Recovery_Days",
            "Recovered_%",
            "Open"
        ]
    ]


episodes_df = build_all_correction_episodes(
    df_nav
)


print()
print("=" * 110)
print("SECTION 2 — CORRECTION EPISODE SUMMARY")
print("=" * 110)

episode_summary = (
    episodes_df
    .groupby("Scheme_Code")
    .agg(
        Completed_Episodes=(
            "Open",
            lambda x: int((~x).sum())
        ),
        Open_Episodes=(
            "Open",
            "sum"
        ),
        Max_Correction=(
            "Correction_%",
            "max"
        )
    )
    .reset_index()
)

print(
    episode_summary.to_string(
        index=False
    )
)


SECTION 2 — CORRECTION EPISODE SUMMARY
 Scheme_Code  Completed_Episodes  Open_Episodes  Max_Correction
      118825                  33              1       37.441726
      118834                  35              0       36.773423
      118989                  33              0       39.513231
      119063                  36              1       38.359143
      119151                   1              0        3.001365
      119218                  35              1       36.550601
      119544                  30              0       35.119378
      119705                  38              0       40.753372
      119714                   2              0        8.538589
      119727                  31              0       32.738405
      120244                  39              1       47.756704
      120334                  23              0       30.608667
      120578                  32              1       29.848204
      120594                  27              1       35.089584


In [26]:
# =================================================================================================
# SECTION 3
# DIAGNOSTICS 1
#
# CURRENT CORRECTION + CYCLE + HISTORICAL OPPORTUNITIES
# =================================================================================================


def get_current_state(
    fund_df,
    episodes
):

    fund_df = (
        fund_df
        .sort_values("Date")
        .reset_index(drop=True)
    )

    current_date = (
        fund_df["Date"].iloc[-1]
    )

    current_nav = float(
        fund_df["NAV"].iloc[-1]
    )

    # ---------------------------------------------------------------------------------------------
    # Current confirmed peak
    #
    # Use the peak of the current active cycle.
    # ---------------------------------------------------------------------------------------------

    open_episodes = (
        episodes[
            episodes["Open"] == True
        ]
        if not episodes.empty
        else pd.DataFrame()
    )

    if not open_episodes.empty:

        current_episode = (
            open_episodes
            .iloc[-1]
        )

        peak_nav = float(
            current_episode["Peak_NAV"]
        )

        peak_date = pd.Timestamp(
            current_episode["Peak_Date"]
        )

        trough_nav = float(
            current_episode["Trough_NAV"]
        )

        trough_date = pd.Timestamp(
            current_episode["Trough_Date"]
        )

        current_correction = (
            (peak_nav - current_nav)
            / peak_nav
            * 100
        )

        if peak_nav > trough_nav:

            recovered = (
                (current_nav - trough_nav)
                /
                (peak_nav - trough_nav)
                * 100
            )

        else:
            recovered = 0

        if current_nav <= trough_nav:

            cycle_phase = (
                "MOVING_TOWARD_TROUGH"
            )

        elif current_nav < peak_nav:

            cycle_phase = (
                "RECOVERING_TOWARD_PEAK"
            )

        else:

            cycle_phase = (
                "AT_OR_ABOVE_PEAK"
            )

        return {

            "Current_Date":
                current_date,

            "Current_NAV":
                current_nav,

            "Peak_NAV":
                peak_nav,

            "Peak_Date":
                peak_date,

            "Trough_NAV":
                trough_nav,

            "Trough_Date":
                trough_date,

            "Current_Correction_%":
                current_correction,

            "Days_From_Peak":
                (
                    current_date
                    - peak_date
                ).days,

            "Recovered_%":
                recovered,

            "Cycle_Phase":
                cycle_phase,

            "Open_Correction":
                True
        }

    # ---------------------------------------------------------------------------------------------
    # No open correction: current NAV is at / near peak.
    # ---------------------------------------------------------------------------------------------

    peak_idx = fund_df["NAV"].idxmax()

    peak_nav = float(
        fund_df.loc[
            peak_idx,
            "NAV"
        ]
    )

    peak_date = pd.Timestamp(
        fund_df.loc[
            peak_idx,
            "Date"
        ]
    )

    current_correction = (
        (peak_nav - current_nav)
        / peak_nav
        * 100
    )

    return {

        "Current_Date":
            current_date,

        "Current_NAV":
            current_nav,

        "Peak_NAV":
            peak_nav,

        "Peak_Date":
            peak_date,

        "Trough_NAV":
            np.nan,

        "Trough_Date":
            pd.NaT,

        "Current_Correction_%":
            current_correction,

        "Days_From_Peak":
            (
                current_date
                - peak_date
            ).days,

        "Recovered_%":
            np.nan,

        "Cycle_Phase":
            "AT_OR_NEAR_PEAK",

        "Open_Correction":
            False
    }


# -------------------------------------------------------------------------------------------------
# HISTORICAL PERCENTILES
# -------------------------------------------------------------------------------------------------

def correction_percentiles(
    completed_episodes
):

    if completed_episodes.empty:

        return {
            "P50": np.nan,
            "P75": np.nan,
            "P90": np.nan,
            "P95": np.nan,
            "N": 0
        }

    values = (
        completed_episodes[
            "Correction_%"
        ]
        .dropna()
    )

    if len(values) == 0:

        return {
            "P50": np.nan,
            "P75": np.nan,
            "P90": np.nan,
            "P95": np.nan,
            "N": 0
        }

    return {

        "P50":
            values.quantile(0.50),

        "P75":
            values.quantile(0.75),

        "P90":
            values.quantile(0.90),

        "P95":
            values.quantile(0.95),

        "N":
            len(values)
    }


# -------------------------------------------------------------------------------------------------
# CORRECTION ZONE
# -------------------------------------------------------------------------------------------------

def correction_zone(
    current_correction,
    p50,
    p75,
    p90,
    p95
):

    if pd.isna(current_correction):
        return "UNKNOWN"

    if pd.isna(p50):
        return "INSUFFICIENT_HISTORY"

    if current_correction < p50:
        return "BELOW_MEDIAN"

    if current_correction < p75:
        return "MEDIAN → P75"

    if current_correction < p90:
        return "P75 → P90"

    if current_correction < p95:
        return "P90 → P95"

    return "ABOVE_P95"


# -------------------------------------------------------------------------------------------------
# HISTORICAL OPPORTUNITY ANALYSIS
# -------------------------------------------------------------------------------------------------

def historical_opportunity_analysis(
    current_state,
    completed_episodes
):

    if completed_episodes.empty:

        return {
            "P50": np.nan,
            "P75": np.nan,
            "P90": np.nan,
            "P95": np.nan,
            "Historical_Episodes": 0,
            "Deeper_Than_Today": 0,
            "Near_Today": 0,
            "Best_Historical_Correction": np.nan,
            "Best_Historical_Date": pd.NaT
        }, pd.DataFrame()

    stats = correction_percentiles(
        completed_episodes
    )

    current_correction = (
        current_state[
            "Current_Correction_%"
        ]
    )

    historical = completed_episodes.copy()

    historical["Comparison"] = np.where(

        historical["Correction_%"]
        >= current_correction,

        "DEEPER_THAN_TODAY",

        np.where(

            historical["Correction_%"]
            >=
            current_correction
            - NEAR_TODAY_TOLERANCE,

            "NEAR_TODAY",

            "SHALLOWER"
        )
    )

    comparable = historical[
        historical["Comparison"].isin(
            [
                "DEEPER_THAN_TODAY",
                "NEAR_TODAY"
            ]
        )
    ].copy()

    if comparable.empty:

        best_correction = np.nan
        best_date = pd.NaT

    else:

        best = comparable.sort_values(
            "Correction_%",
            ascending=False
        ).iloc[0]

        best_correction = (
            best["Correction_%"]
        )

        best_date = (
            best["Trough_Date"]
        )

    result = {

        "P50":
            stats["P50"],

        "P75":
            stats["P75"],

        "P90":
            stats["P90"],

        "P95":
            stats["P95"],

        "Historical_Episodes":
            stats["N"],

        "Deeper_Than_Today":
            int(
                (
                    historical[
                        "Correction_%"
                    ]
                    >= current_correction
                ).sum()
            ),

        "Near_Today":
            int(
                (
                    historical["Comparison"]
                    == "NEAR_TODAY"
                ).sum()
            ),

        "Best_Historical_Correction":
            best_correction,

        "Best_Historical_Date":
            best_date
    }

    return result, comparable


# -------------------------------------------------------------------------------------------------
# CURRENT CORRECTION SIGNAL
# -------------------------------------------------------------------------------------------------

def correction_signal(
    current_correction,
    p50,
    p75,
    p90,
    p95
):

    if pd.isna(current_correction):
        return "NORMAL"

    if pd.isna(p50):
        return "NORMAL"

    if current_correction >= p95:
        return "STRONG ACCUMULATE"

    if current_correction >= p90:
        return "STRONG ACCUMULATE"

    if current_correction >= p75:
        return "ACCUMULATE"

    if current_correction >= p50:
        return "WATCH"

    return "NORMAL"


# -------------------------------------------------------------------------------------------------
# BUILD DIAGNOSTICS 1 FOR ALL FUNDS
# -------------------------------------------------------------------------------------------------

def build_diagnostics1(
    df_nav,
    episodes_df
):

    rows = []

    for code, fund_df in df_nav.groupby(
        "Scheme_Code"
    ):

        fund_episodes = episodes_df[
            episodes_df["Scheme_Code"] == code
        ].copy()

        current = get_current_state(
            fund_df,
            fund_episodes
        )

        completed = fund_episodes[
            fund_episodes["Open"] == False
        ].copy()

        hist, comparable = (
            historical_opportunity_analysis(
                current,
                completed
            )
        )

        signal = correction_signal(
            current[
                "Current_Correction_%"
            ],
            hist["P50"],
            hist["P75"],
            hist["P90"],
            hist["P95"]
        )

        zone = correction_zone(
            current[
                "Current_Correction_%"
            ],
            hist["P50"],
            hist["P75"],
            hist["P90"],
            hist["P95"]
        )

        rows.append({

            "Scheme_Code":
                code,

            "Scheme_Name":
                fund_df["Scheme_Name"].iloc[0],

            "Current_NAV":
                current["Current_NAV"],

            "Peak_NAV":
                current["Peak_NAV"],

            "Peak_Date":
                current["Peak_Date"],

            "Current_Correction_%":
                current[
                    "Current_Correction_%"
                ],

            "Correction_Zone":
                zone,

            "Cycle_Phase":
                current[
                    "Cycle_Phase"
                ],

            "Recovered_%":
                current[
                    "Recovered_%"
                ],

            "Days_From_Peak":
                current[
                    "Days_From_Peak"
                ],

            "P50":
                hist["P50"],

            "P75":
                hist["P75"],

            "P90":
                hist["P90"],

            "P95":
                hist["P95"],

            "Historical_Episodes":
                hist[
                    "Historical_Episodes"
                ],

            "Deeper_Than_Today":
                hist[
                    "Deeper_Than_Today"
                ],

            "Near_Today":
                hist["Near_Today"],

            "Best_Historical_Correction":
                hist[
                    "Best_Historical_Correction"
                ],

            "Best_Historical_Date":
                hist[
                    "Best_Historical_Date"
                ],

            "Correction_Signal":
                signal
        })

    return pd.DataFrame(rows)


diagnostics1_df = build_diagnostics1(
    df_nav,
    episodes_df
)


print()
print("=" * 110)
print("SECTION 3 — DIAGNOSTICS 1")
print("=" * 110)

print(
    diagnostics1_df[
        [
            "Scheme_Code",
            "Scheme_Name",
            "Current_Correction_%",
            "Correction_Zone",
            "Cycle_Phase",
            "Recovered_%",
            "P75",
            "P90",
            "P95",
            "Deeper_Than_Today",
            "Correction_Signal"
        ]
    ]
    .sort_values(
        "Current_Correction_%",
        ascending=False
    )
    .to_string(index=False)
)


SECTION 3 — DIAGNOSTICS 1
 Scheme_Code                                                               Scheme_Name  Current_Correction_% Correction_Zone            Cycle_Phase  Recovered_%       P75       P90       P95  Deeper_Than_Today Correction_Signal
      120594                  ICICI Prudential Technology Fund - Direct Plan -  Growth             17.876715       P90 → P95 RECOVERING_TOWARD_PEAK    36.102373 12.581792 16.323011 25.481032                  2 STRONG ACCUMULATE
      152430                        HDFC NIFTY200 Momentum 30 Index Fund - Direct Plan             17.510783       ABOVE_P95 RECOVERING_TOWARD_PEAK    45.019871  7.230951  8.285459  9.033103                  0 STRONG ACCUMULATE
      120578                  SBI TECHNOLOGY OPPORTUNITIES FUND - DIRECT PLAN - GROWTH             12.550009       P75 → P90 RECOVERING_TOWARD_PEAK    48.236936  9.316319 17.420976 23.408808                  6        ACCUMULATE
      138528 PGIM India Global Equity Opportunities Fund of F

In [27]:
# =================================================================================================
# SECTION 4
# DIAGNOSTICS 2
#
# A. HISTORICAL-TROUGH GROWTH ENGINE
# B. PEAK / OVERSHOOT / PROFIT-TAKING ENGINE
# C. STRATEGIC GROUP OVERLAP
# =================================================================================================


# -------------------------------------------------------------------------------------------------
# A. HISTORICAL-TROUGH GROWTH
# -------------------------------------------------------------------------------------------------

def historical_trough_growth(
    fund_df,
    episodes_df
):

    fund_df = (
        fund_df
        .sort_values("Date")
        .reset_index(drop=True)
    )

    current_date = (
        fund_df["Date"].iloc[-1]
    )

    current_nav = float(
        fund_df["NAV"].iloc[-1]
    )

    completed = episodes_df[
        episodes_df["Open"] == False
    ].copy()

    if completed.empty:

        return {
            "Historical_Troughs": 0,
            "Best_Trough_Growth_%": np.nan,
            "Best_Trough_Date": pd.NaT,
            "Median_Trough_Growth_%": np.nan
        }

    # Only use troughs older than 6 months.
    completed["Trough_Age_Days"] = (
        current_date
        - completed["Trough_Date"]
    ).dt.days

    eligible = completed[
        completed["Trough_Age_Days"]
        >= MIN_TROUGH_AGE_DAYS
    ].copy()

    if eligible.empty:

        return {
            "Historical_Troughs": 0,
            "Best_Trough_Growth_%": np.nan,
            "Best_Trough_Date": pd.NaT,
            "Median_Trough_Growth_%": np.nan
        }

    eligible["Growth_From_Trough_%"] = (

        (
            current_nav
            / eligible["Trough_NAV"]
        )
        - 1
    ) * 100

    best = eligible.sort_values(
        "Growth_From_Trough_%",
        ascending=False
    ).iloc[0]

    return {

        "Historical_Troughs":
            len(eligible),

        "Best_Trough_Growth_%":
            best[
                "Growth_From_Trough_%"
            ],

        "Best_Trough_Date":
            best[
                "Trough_Date"
            ],

        "Median_Trough_Growth_%":
            eligible[
                "Growth_From_Trough_%"
            ].median()
    }


# -------------------------------------------------------------------------------------------------
# B. PROFIT-TAKING / OVERSHOOT ENGINE
#
# Each new significant peak is compared against the previous significant peak.
#
# Example:
#
# Previous peak = 100
# New peak      = 125
#
# Overshoot = +25%
#
# This is independent of correction.
# -------------------------------------------------------------------------------------------------

def build_overshoot_episodes(
    fund_df,
    correction_episodes
):

    if correction_episodes.empty:
        return pd.DataFrame()

    completed = correction_episodes[
        correction_episodes["Open"] == False
    ].copy()

    if completed.empty:
        return pd.DataFrame()

    peaks = (
        completed[
            [
                "Peak_Date",
                "Peak_NAV"
            ]
        ]
        .drop_duplicates()
        .sort_values("Peak_Date")
        .reset_index(drop=True)
    )

    if len(peaks) < 2:
        return pd.DataFrame()

    rows = []

    for i in range(1, len(peaks)):

        previous_peak = peaks.iloc[i - 1]
        new_peak = peaks.iloc[i]

        previous_nav = float(
            previous_peak["Peak_NAV"]
        )

        new_nav = float(
            new_peak["Peak_NAV"]
        )

        if previous_nav <= 0:
            continue

        overshoot = (
            (new_nav / previous_nav)
            - 1
        ) * 100

        rows.append({

            "Previous_Peak_Date":
                previous_peak["Peak_Date"],

            "Previous_Peak_NAV":
                previous_nav,

            "New_Peak_Date":
                new_peak["Peak_Date"],

            "New_Peak_NAV":
                new_nav,

            "Overshoot_%":
                overshoot,

            "Peak_to_New_Peak_Days":
                (
                    new_peak["Peak_Date"]
                    - previous_peak["Peak_Date"]
                ).days
        })

    return pd.DataFrame(rows)


# -------------------------------------------------------------------------------------------------
# PROFIT-TAKING SIGNAL
# -------------------------------------------------------------------------------------------------

def profit_signal(
    current_nav,
    historical_peaks
):

    if historical_peaks.empty:
        return "NO PROFIT SIGNAL"

    previous_peaks = historical_peaks[
        historical_peaks["New_Peak_NAV"]
        < current_nav
    ].copy()

    if previous_peaks.empty:
        return "NO PROFIT SIGNAL"

    max_previous_peak = (
        previous_peaks["New_Peak_NAV"]
        .max()
    )

    overshoot = (
        (current_nav / max_previous_peak)
        - 1
    ) * 100

    # Conservative profit-taking thresholds
    if overshoot >= 25:
        return "STRONG TRIM"

    if overshoot >= 15:
        return "TRIM"

    if overshoot >= 10:
        return "WATCH PROFIT"

    return "NO PROFIT SIGNAL"


# -------------------------------------------------------------------------------------------------
# STRATEGIC GROUP DEFINITIONS
#
# HDFC Nifty 50 and ICICI Nifty 50 are intentionally in the SAME strategic group.
#
# Fund-house differences do NOT create separate strategic exposures.
# -------------------------------------------------------------------------------------------------

strategic_groups = {

    "CORE_EQUITY": [

        119063,     # HDFC Nifty 50
        120620,     # ICICI Prudential Nifty 50

        143341,     # UTI Nifty Next 50

        120152,     # Kotak Large Cap
        120465,     # Axis Large Cap
        118825,     # Mirae Asset Large Cap

        103819,     # DSP Large & Mid Cap - Regular
        119218,     # DSP Large & Mid Cap - Direct

        112932,     # Mirae Asset Large & Midcap - Regular
        118834,     # Mirae Asset Large & Midcap - Direct
    ],

    "GROWTH_EQUITY": [

        120492,     # JM Flexicap
        122639,     # Parag Parikh Flexi Cap
        141226,     # Mahindra Manulife Multi Cap

        120505,     # Axis Midcap
        142110,     # Mahindra Manulife Mid Cap
        119775,     # Kotak Midcap

        118989,     # HDFC Mid Cap
        127042,     # Motilal Oswal Midcap

        125354,     # Axis Small Cap
        119556,     # ABSL Small Cap
        153196,     # Mirae Asset Small Cap
        130503,     # HDFC Small Cap
    ],

    "FACTOR_EQUITY": [

        152430,     # HDFC Nifty 200 Momentum 30
        148703,     # UTI Nifty 200 Momentum 30
    ],

    "SECTOR_EQUITY": [

        109445,     # ICICI Prudential Banking & Financial Services
        120244,     # ICICI Prudential Banking & Financial Services Direct
        120733,     # UTI Banking & Financial Services
        148623,     # Mirae Asset Banking & Financial Services

        120594,     # ICICI Prudential Technology
        120578,     # SBI Technology Opportunities
        150597,     # Mirae Asset Global X AI & Technology

        143783,     # Mirae Asset Healthcare
        147409,     # ABSL Pharma & Healthcare
        145454,     # DSP Healthcare

        120728,     # UTI Infrastructure
        120351,     # LIC MF Infrastructure
        133516,     # ABSL Manufacturing
        119705,     # SBI COMMA
    ],

    "INTERNATIONAL": [

        138528,
        140243,
        145552,
        119277,
    ],

    "HYBRID_DEBT_OTHER": [

        135781,     # Mirae Asset ELSS
        119544,     # ABSL ELSS

        119727,     # SBI Focused Fund

        120334,     # ICICI Prudential Multi-Asset
    ],

    "DEBT": [

        119714,     # SBI Medium to Long Duration
    ]
}


# -------------------------------------------------------------------------------------------------
# REVERSE MAP:
# SCHEME CODE -> STRATEGIC GROUP
# -------------------------------------------------------------------------------------------------

code_to_group = {}

for group, codes_in_group in strategic_groups.items():

    for code in codes_in_group:

        code_to_group[code] = group


# -------------------------------------------------------------------------------------------------
# GROUP OVERLAP / PRESSURE
# -------------------------------------------------------------------------------------------------

def group_overlap_analysis(
    diagnostics_df,
    group_map
):

    df = diagnostics_df.copy()

    df["Strategic_Group"] = (
        df["Scheme_Code"]
        .map(group_map)
        .fillna("UNCLASSIFIED")
    )

    # Buy pressure
    signal_score = {

        "STRONG ACCUMULATE": 4,
        "ACCUMULATE": 3,
        "WATCH": 1,
        "NORMAL": 0
    }

    df["Buy_Score"] = (
        df["Correction_Signal"]
        .map(signal_score)
        .fillna(0)
    )

    group_summary = (
        df
        .groupby("Strategic_Group")
        .agg(

            Funds=(
                "Scheme_Code",
                "count"
            ),

            Buy_Pressure=(
                "Buy_Score",
                "sum"
            ),

            Max_Correction=(
                "Current_Correction_%",
                "max"
            ),

            Average_Correction=(
                "Current_Correction_%",
                "mean"
            )
        )
        .reset_index()
    )

    # Normalize buy pressure
    group_summary["Normalized_Buy_Pressure"] = (

        group_summary["Buy_Pressure"]
        /
        group_summary["Funds"].clip(lower=1)
    )

    return df, group_summary


# -------------------------------------------------------------------------------------------------
# BUILD DIAGNOSTICS 2
# -------------------------------------------------------------------------------------------------

def build_diagnostics2(
    df_nav,
    episodes_df,
    diagnostics1_df
):

    rows = []

    for code, fund_df in df_nav.groupby(
        "Scheme_Code"
    ):

        fund_episodes = episodes_df[
            episodes_df["Scheme_Code"] == code
        ].copy()

        current_nav = float(
            fund_df["NAV"].iloc[-1]
        )

        # Historical trough growth
        trough = historical_trough_growth(
            fund_df,
            fund_episodes
        )

        # Overshoot history
        overshoots = build_overshoot_episodes(
            fund_df,
            fund_episodes
        )

        current_profit_signal = (
            profit_signal(
                current_nav,
                overshoots
            )
        )

        # Current overshoot over most recent
        # previous significant peak
        current_overshoot = np.nan

        if not overshoots.empty:

            previous_peaks = overshoots[
                overshoots[
                    "New_Peak_NAV"
                ] < current_nav
            ]

            if not previous_peaks.empty:

                reference_peak = (
                    previous_peaks[
                        "New_Peak_NAV"
                    ].max()
                )

                current_overshoot = (

                    (
                        current_nav
                        / reference_peak
                    )
                    - 1
                ) * 100

        rows.append({

            "Scheme_Code":
                code,

            "Scheme_Name":
                fund_df[
                    "Scheme_Name"
                ].iloc[0],

            "Historical_Troughs":
                trough[
                    "Historical_Troughs"
                ],

            "Best_Trough_Growth_%":
                trough[
                    "Best_Trough_Growth_%"
                ],

            "Best_Trough_Date":
                trough[
                    "Best_Trough_Date"
                ],

            "Median_Trough_Growth_%":
                trough[
                    "Median_Trough_Growth_%"
                ],

            "Historical_Peak_Transitions":
                len(overshoots),

            "Current_Overshoot_%":
                current_overshoot,

            "Profit_Signal":
                current_profit_signal
        })

    return pd.DataFrame(rows)


diagnostics2_df = build_diagnostics2(
    df_nav,
    episodes_df,
    diagnostics1_df
)


# -------------------------------------------------------------------------------------------------
# MERGE DIAGNOSTICS
# -------------------------------------------------------------------------------------------------

diagnostics_df = (
    diagnostics1_df
    .merge(
        diagnostics2_df,
        on=[
            "Scheme_Code",
            "Scheme_Name"
        ],
        how="left"
    )
)


# -------------------------------------------------------------------------------------------------
# ADD STRATEGIC GROUP
# -------------------------------------------------------------------------------------------------

diagnostics_df[
    "Strategic_Group"
] = (
    diagnostics_df[
        "Scheme_Code"
    ]
    .map(code_to_group)
    .fillna("UNCLASSIFIED")
)


# -------------------------------------------------------------------------------------------------
# GROUP OVERLAP
# -------------------------------------------------------------------------------------------------

diagnostics_df, group_summary_df = (
    group_overlap_analysis(
        diagnostics_df,
        code_to_group
    )
)


print()
print("=" * 110)
print("SECTION 4 — DIAGNOSTICS 2")
print("=" * 110)

print(
    diagnostics_df[
        [
            "Scheme_Code",
            "Scheme_Name",
            "Strategic_Group",
            "Best_Trough_Growth_%",
            "Current_Overshoot_%",
            "Profit_Signal"
        ]
    ]
    .sort_values(
        "Best_Trough_Growth_%",
        ascending=False
    )
    .to_string(index=False)
)


print()
print("=" * 110)
print("STRATEGIC GROUP OVERLAP")
print("=" * 110)

print(
    group_summary_df
    .sort_values(
        "Normalized_Buy_Pressure",
        ascending=False
    )
    .to_string(index=False)
)


SECTION 4 — DIAGNOSTICS 2
 Scheme_Code                                                               Scheme_Name   Strategic_Group  Best_Trough_Growth_%  Current_Overshoot_%    Profit_Signal
      118834                    Mirae Asset Large & Midcap Fund - Direct Plan - Growth       CORE_EQUITY           1465.033696             0.809764 NO PROFIT SIGNAL
      118989                           HDFC Mid Cap Fund - Growth Option - Direct Plan     GROWTH_EQUITY           1427.819694             4.288286 NO PROFIT SIGNAL
      125354                                Axis Small Cap Fund - Direct Plan - Growth     GROWTH_EQUITY           1309.499489             8.371947 NO PROFIT SIGNAL
      120594                  ICICI Prudential Technology Fund - Direct Plan -  Growth     SECTOR_EQUITY            972.062663             2.088513 NO PROFIT SIGNAL
      120578                  SBI TECHNOLOGY OPPORTUNITIES FUND - DIRECT PLAN - GROWTH     SECTOR_EQUITY            929.875424             0.400624 

In [28]:
# =================================================================================================
# SECTION 5
# INVESTMENT ENGINE
#
# Converts diagnostics into a suggested allocation of ₹2,00,000.
#
# This is NOT a "buy every fund" engine.
#
# It deliberately applies strategic-group overlap penalties.
# =================================================================================================


# -------------------------------------------------------------------------------------------------
# SIGNAL SCORES
# -------------------------------------------------------------------------------------------------

BUY_SIGNAL_SCORE = {

    "NORMAL":
        0,

    "WATCH":
        1,

    "ACCUMULATE":
        3,

    "STRONG ACCUMULATE":
        5
}


# -------------------------------------------------------------------------------------------------
# CYCLE PHASE SCORES
#
# Important distinction:
#
# MOVING_TOWARD_TROUGH
#     Strongest opportunity if correction is becoming deeper.
#
# RECOVERING_TOWARD_PEAK
#     Still investable, but urgency declines as recovery progresses.
#
# AT_OR_NEAR_PEAK
#     Do not chase merely because the fund is historically good.
# -------------------------------------------------------------------------------------------------

def cycle_score(
    row
):

    phase = row[
        "Cycle_Phase"
    ]

    correction = row[
        "Current_Correction_%"
    ]

    recovered = row[
        "Recovered_%"
    ]

    if phase == "MOVING_TOWARD_TROUGH":

        return 1.20

    if phase == "RECOVERING_TOWARD_PEAK":

        if pd.isna(recovered):
            return 1.00

        if recovered < 30:
            return 1.10

        if recovered < 60:
            return 1.00

        if recovered < 80:
            return 0.80

        return 0.50

    return 0.25


# -------------------------------------------------------------------------------------------------
# DEPTH SCORE
#
# Current correction relative to historical percentiles.
# -------------------------------------------------------------------------------------------------

def depth_score(
    row
):

    correction = row[
        "Current_Correction_%"
    ]

    p75 = row["P75"]
    p90 = row["P90"]
    p95 = row["P95"]

    if pd.isna(correction):
        return 0

    if pd.notna(p95) and correction >= p95:
        return 5

    if pd.notna(p90) and correction >= p90:
        return 4

    if pd.notna(p75) and correction >= p75:
        return 3

    return 1


# -------------------------------------------------------------------------------------------------
# HISTORICAL OPPORTUNITY SCORE
#
# How many historical episodes were as deep or deeper?
#
# More historical confirmation = greater robustness.
# -------------------------------------------------------------------------------------------------

def robustness_score(
    row
):

    deeper = row[
        "Deeper_Than_Today"
    ]

    historical = row[
        "Historical_Episodes"
    ]

    if historical <= 0:
        return 0.0

    ratio = (
        deeper / historical
    )

    # A current correction that is among the deepest
    # historical episodes receives higher score.
    if ratio <= 0.10:
        return 3.0

    if ratio <= 0.25:
        return 2.0

    if ratio <= 0.50:
        return 1.0

    return 0.25


# -------------------------------------------------------------------------------------------------
# GROUP OVERLAP PENALTY
#
# If 4 banking funds are simultaneously attractive,
# we do NOT want to allocate four independent full positions.
#
# The group receives a shared capital pool.
# -------------------------------------------------------------------------------------------------

def calculate_group_penalty(
    diagnostics_df
):

    df = diagnostics_df.copy()

    attractive = df[
        df["Correction_Signal"].isin(
            [
                "ACCUMULATE",
                "STRONG ACCUMULATE"
            ]
        )
    ].copy()

    counts = (
        attractive
        .groupby("Strategic_Group")
        .size()
        .to_dict()
    )

    df["Group_Attractive_Count"] = (
        df["Strategic_Group"]
        .map(counts)
        .fillna(0)
    )

    # More funds expressing the same theme =>
    # smaller individual allocation.
    df["Overlap_Factor"] = np.where(

        df["Group_Attractive_Count"] <= 1,

        1.00,

        1
        /
        np.sqrt(
            df["Group_Attractive_Count"]
        )
    )

    return df


# -------------------------------------------------------------------------------------------------
# PROFIT-TAKING PENALTY
#
# A fund that is simultaneously near a profit-taking condition
# should not receive new capital aggressively.
# -------------------------------------------------------------------------------------------------

def profit_penalty(
    row
):

    signal = row[
        "Profit_Signal"
    ]

    if signal == "STRONG TRIM":
        return 0.00

    if signal == "TRIM":
        return 0.15

    if signal == "WATCH PROFIT":
        return 0.50

    return 1.00


# -------------------------------------------------------------------------------------------------
# BUILD INVESTMENT SCORE
# -------------------------------------------------------------------------------------------------

def build_investment_engine(
    diagnostics_df,
    total_amount=TOTAL_MONTHLY_INVESTMENT
):

    df = diagnostics_df.copy()

    df = calculate_group_penalty(
        df
    )

    # Base signal
    df["Signal_Score"] = (
        df[
            "Correction_Signal"
        ]
        .map(
            BUY_SIGNAL_SCORE
        )
        .fillna(0)
    )

    # Historical depth
    df["Depth_Score"] = (
        df.apply(
            depth_score,
            axis=1
        )
    )

    # Historical robustness
    df["Robustness_Score"] = (
        df.apply(
            robustness_score,
            axis=1
        )
    )

    # Cycle
    df["Cycle_Score"] = (
        df.apply(
            cycle_score,
            axis=1
        )
    )

    # Base pressure
    df["Buy_Pressure"] = (

        df["Signal_Score"]
        *
        (
            0.50
            +
            0.20
            *
            df["Depth_Score"]
            /
            5
        )
        *
        (
            0.75
            +
            0.25
            *
            df["Robustness_Score"]
            /
            3
        )
        *
        df["Cycle_Score"]
    )

    # Strategic overlap
    df["Buy_Pressure"] *= (
        df["Overlap_Factor"]
    )

    # Profit-taking penalty
    df["Profit_Factor"] = (
        df.apply(
            profit_penalty,
            axis=1
        )
    )

    df["Buy_Pressure"] *= (
        df["Profit_Factor"]
    )

    # ---------------------------------------------------------------------------------------------
    # Only funds with meaningful buy signals receive capital.
    # ---------------------------------------------------------------------------------------------

    eligible = df[
        df["Correction_Signal"].isin(
            [
                "ACCUMULATE",
                "STRONG ACCUMULATE"
            ]
        )
        &
        (df["Buy_Pressure"] > 0)
    ].copy()

    # ---------------------------------------------------------------------------------------------
    # Normalize capital
    # ---------------------------------------------------------------------------------------------

    total_pressure = (
        eligible[
            "Buy_Pressure"
        ].sum()
    )

    df["Investment_₹"] = 0.0

    if total_pressure > 0:

        df.loc[
            eligible.index,
            "Investment_₹"
        ] = (

            eligible[
                "Buy_Pressure"
            ]
            /
            total_pressure
            *
            total_amount
        )

    # Round to nearest ₹1,000
    df["Investment_₹"] = (
        df["Investment_₹"]
        .div(1000)
        .round()
        .mul(1000)
    )

    return df


investment_df = build_investment_engine(
    diagnostics_df,
    TOTAL_MONTHLY_INVESTMENT
)


# -------------------------------------------------------------------------------------------------
# FINAL INVESTMENT OUTPUT
# -------------------------------------------------------------------------------------------------

investment_output = (
    investment_df[
        [
            "Scheme_Code",
            "Scheme_Name",
            "Strategic_Group",

            "Correction_Signal",
            "Current_Correction_%",
            "Correction_Zone",
            "Cycle_Phase",
            "Recovered_%",

            "P75",
            "P90",
            "P95",

            "Historical_Episodes",
            "Deeper_Than_Today",

            "Best_Trough_Growth_%",

            "Profit_Signal",
            "Current_Overshoot_%",

            "Group_Attractive_Count",
            "Overlap_Factor",

            "Buy_Pressure",
            "Investment_₹"
        ]
    ]
    .sort_values(
        [
            "Investment_₹",
            "Buy_Pressure"
        ],
        ascending=False
    )
)


print()
print("=" * 130)
print("SECTION 5 — ₹2,00,000 INVESTMENT ENGINE")
print("=" * 130)

print(
    investment_output.to_string(
        index=False
    )
)


print()
print("=" * 130)
print("INVESTMENT SUMMARY")
print("=" * 130)

print(
    f"Total requested : ₹{TOTAL_MONTHLY_INVESTMENT:,.0f}"
)

print(
    f"Total allocated : ₹{investment_output['Investment_₹'].sum():,.0f}"
)

print(
    f"Funds receiving allocation : "
    f"{(
        investment_output[
            'Investment_₹'
        ] > 0
    ).sum()}"
)


SECTION 5 — ₹2,00,000 INVESTMENT ENGINE
 Scheme_Code                                                               Scheme_Name   Strategic_Group Correction_Signal  Current_Correction_% Correction_Zone            Cycle_Phase  Recovered_%       P75       P90       P95  Historical_Episodes  Deeper_Than_Today  Best_Trough_Growth_%    Profit_Signal  Current_Overshoot_%  Group_Attractive_Count  Overlap_Factor  Buy_Pressure  Investment_₹
      152430                        HDFC NIFTY200 Momentum 30 Index Fund - Direct Plan     FACTOR_EQUITY STRONG ACCUMULATE             17.510783       ABOVE_P95 RECOVERING_TOWARD_PEAK    45.019871  7.230951  8.285459  9.033103                    7                  0              7.956684 NO PROFIT SIGNAL                  NaN                     1.0        1.000000      3.500000       79000.0
      120594                  ICICI Prudential Technology Fund - Direct Plan -  Growth     SECTOR_EQUITY STRONG ACCUMULATE             17.876715       P90 → P95 RECOVERI

In [33]:
# =================================================================================================
# SECTION 7
# HISTORICAL ROBUSTNESS TEST
# =================================================================================================

def show_historical_buying_opportunities(
    scheme_code
):

    fund_df = df_nav[
        df_nav["Scheme_Code"]
        == scheme_code
    ].copy()

    fund_episodes = episodes_df[
        episodes_df["Scheme_Code"]
        == scheme_code
    ].copy()

    if fund_df.empty:
        print(
            f"Scheme {scheme_code} not found."
        )
        return

    current = get_current_state(
        fund_df,
        fund_episodes
    )

    completed = fund_episodes[
        fund_episodes["Open"] == False
    ].copy()

    hist, comparable = (
        historical_opportunity_analysis(
            current,
            completed
        )
    )

    print()
    print("=" * 120)
    print(
        f"HISTORICAL BUYING OPPORTUNITY TEST — "
        f"{fund_df['Scheme_Name'].iloc[0]}"
    )
    print("=" * 120)

    print(
        f"Current NAV          : "
        f"{current['Current_NAV']:.4f}"
    )

    print(
        f"Current correction    : "
        f"{current['Current_Correction_%']:.2f}%"
    )

    print(
        f"Historical P75        : "
        f"{hist['P75']:.2f}%"
        if pd.notna(hist["P75"])
        else
        "Historical P75        : N/A"
    )

    print(
        f"Historical P90        : "
        f"{hist['P90']:.2f}%"
        if pd.notna(hist["P90"])
        else
        "Historical P90        : N/A"
    )

    print(
        f"Historical P95        : "
        f"{hist['P95']:.2f}%"
        if pd.notna(hist["P95"])
        else
        "Historical P95        : N/A"
    )

    print()

    if comparable.empty:

        print(
            "No comparable historical episodes."
        )

        return

    columns = [

        "Peak_Date",
        "Peak_NAV",

        "Trough_Date",
        "Trough_NAV",

        "Recovery_Date",

        "Correction_%",

        "Peak_to_Trough_Days",

        "Trough_to_Recovery_Days",

        "Total_Recovery_Days",

        "Comparison"
    ]

    print(
        comparable[
            columns
        ]
        .sort_values(
            "Correction_%",
            ascending=False
        )
        .to_string(
            index=False
        )
    )


# Example:
#
show_historical_buying_opportunities(120594)
show_historical_buying_opportunities(152430)
show_historical_buying_opportunities(127042)


HISTORICAL BUYING OPPORTUNITY TEST — ICICI Prudential Technology Fund - Direct Plan -  Growth
Current NAV          : 205.3000
Current correction    : 17.88%
Historical P75        : 12.58%
Historical P90        : 16.32%
Historical P95        : 25.48%

 Peak_Date  Peak_NAV Trough_Date  Trough_NAV Recovery_Date  Correction_%  Peak_to_Trough_Days  Trough_to_Recovery_Days  Total_Recovery_Days        Comparison
2020-02-19     65.86  2020-03-23       42.75    2020-07-16     35.089584                   33                    115.0                148.0 DEEPER_THAN_TODAY
2022-01-04    189.26  2022-06-17      134.11    2024-01-12     29.139808                  164                    574.0                738.0 DEEPER_THAN_TODAY
2015-08-19     45.09  2016-11-21       37.45    2017-11-09     16.943890                  460                    353.0                813.0        NEAR_TODAY

HISTORICAL BUYING OPPORTUNITY TEST — HDFC NIFTY200 Momentum 30 Index Fund - Direct Plan
Current NAV          : 10.5

In [34]:
# @title
# =================================================================================================
# SECTION 6
# FINAL DEBUG VIEW
# =================================================================================================

debug_columns = [

    "Scheme_Code",
    "Scheme_Name",
    "Strategic_Group",

    # Current state
    "Current_NAV",
    "Peak_NAV",
    "Peak_Date",
    "Current_Correction_%",
    "Correction_Zone",
    "Cycle_Phase",
    "Recovered_%",

    # Historical correction
    "P50",
    "P75",
    "P90",
    "P95",

    "Historical_Episodes",
    "Deeper_Than_Today",
    "Near_Today",
    "Best_Historical_Correction",
    "Best_Historical_Date",

    # Signals
    "Correction_Signal",

    # Historical trough growth
    "Historical_Troughs",
    "Best_Trough_Growth_%",
    "Best_Trough_Date",
    "Median_Trough_Growth_%",

    # Profit
    "Current_Overshoot_%",
    "Profit_Signal",

    # Group
    "Group_Attractive_Count",
    "Overlap_Factor",

    # Investment
    "Buy_Pressure",
    "Investment_₹"
]


debug_output = (
    investment_df[
        [
            c
            for c in debug_columns
            if c in investment_df.columns
        ]
    ]
    .sort_values(
        [
            "Correction_Signal",
            "Current_Correction_%"
        ],
        ascending=[True, False]
    )
)


print(
    debug_output.to_string(
        index=False
    )
)

 Scheme_Code                                                               Scheme_Name   Strategic_Group  Current_NAV  Peak_NAV  Peak_Date  Current_Correction_% Correction_Zone            Cycle_Phase  Recovered_%      P50       P75       P90       P95  Historical_Episodes  Deeper_Than_Today  Near_Today  Best_Historical_Correction Best_Historical_Date Correction_Signal  Historical_Troughs  Best_Trough_Growth_% Best_Trough_Date  Median_Trough_Growth_%  Current_Overshoot_%    Profit_Signal  Group_Attractive_Count  Overlap_Factor  Buy_Pressure  Investment_₹
      120578                  SBI TECHNOLOGY OPPORTUNITIES FUND - DIRECT PLAN - GROWTH     SECTOR_EQUITY     235.1978  268.9512 2025-12-22             12.550009       P75 → P90 RECOVERING_TOWARD_PEAK    48.236936 5.329085  9.316319 17.420976 23.408808                   32                  6           1                   29.848204           2020-03-23        ACCUMULATE                  32            929.875424       2013-04-26           